# Routing Pattern Hands-On Code Example

This code demonstrates a simple agent-like system using LangChain and Google's Generative AI. It sets up a "coordinator" that routes user requests to different simulated "sub-agent" handlers based on the request's intent (booking, information, or unclear). The system uses a language model to classify the request and then delegates it to the appropriate handler function, simulating a basic delegation pattern often seen in multi-agent architectures.

In [1]:
# !pip install langchain langchain-community langchain-google-genai langgraph

> Note: Create a `.env` file in the same directory with your Google Generative AI API key:
> ```
> GOOGLE_API_KEY="<your_google_api_key_here>"
> ```

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch

In [3]:
load_dotenv(override=True)

True

In [ ]:
# Initialize the Language Model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [5]:
def booking_handler(request: str) -> str:
    """Simulates the Booking Agent handling a request."""
    print("\n--- DELEGATING TO BOOKING HANDLER ---")
    return f"Booking Handler processed request: '{request}'. Result: Simulated booking action."

In [6]:
def info_handler(request: str) -> str:
    """Simulates the Information Agent handling a request."""
    print("\n--- DELEGATING TO INFO HANDLER ---")
    return f"Info Handler processed request: '{request}'. Result: Simulated info retrieval."

In [7]:
def unclear_handler(request: str) -> str:
    """Handles requests that couldn't be delegated."""
    print("\n--- UNCLEAR REQUEST HANDLER ---")
    return f"Coordinator could not delegate request: '{request}'. Result: Please clarify."

In [8]:
# This chain decides which handler to delegate to.
coordinator_router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the user's request and determine which specialist handler should process it.
     - If the request is related to booking flights or hotels, output 'booker'.
     - For all other general information questions, output 'info'.
     - If the request is unclear or doesn't fit either category, output 'unclear'.
     ONLY output one word: 'booker', 'info', or 'unclear'."""),
    ("user", "{request}"),
])

coordinator_router_chain = coordinator_router_prompt | llm | StrOutputParser()

In [9]:
# Define the delegation logic
# Use RunnableBranch to route based on the router chain's output

# Define the branches for the RunnableBranch
branches = {
    "booker": RunnablePassthrough.assign(output=lambda x: booking_handler(x['request']['request'])),
    "info": RunnablePassthrough.assign(output=lambda x: info_handler(x['request']['request'])),
    "unclear": RunnablePassthrough.assign(output=lambda x: unclear_handler(x['request']['request'])),
}

# Create the RunnableBranch. It takes the output of the router chain and routes the original input ('request') to the corresponding handler.
delegation_branch = RunnableBranch(
    (lambda x: x['decision'].strip() == 'booker', branches['booker']),
    (lambda x: x['decision'].strip() == 'info', branches['info']),
    branches['unclear'],
)

# Combine the router chain and delegation branch into a single runnable
coordinator_agent = {
    "decision": coordinator_router_chain,
    "request": RunnablePassthrough(),
} | delegation_branch | (lambda x: x['output'])

In [10]:
print("--- Running with a booking request ---")
request_a = "Book me a flight to London."
result_a = coordinator_agent.invoke({"request": request_a})
print(f"Final Result A: {result_a}")

--- Running with a booking request ---

--- DELEGATING TO BOOKING HANDLER ---
Final Result A: Booking Handler processed request: 'Book me a flight to London.'. Result: Simulated booking action.


In [11]:
print("\n--- Running with an info request ---")
request_b = "What is the capital of Italy?"
result_b = coordinator_agent.invoke({"request": request_b})
print(f"Final Result B: {result_b}")


--- Running with an info request ---

--- DELEGATING TO INFO HANDLER ---
Final Result B: Info Handler processed request: 'What is the capital of Italy?'. Result: Simulated info retrieval.


In [12]:
print("\n--- Running with an unclear request ---")
request_c = "I need help."
result_c = coordinator_agent.invoke({"request": request_c})
print(f"Final Result C: {result_c}")


--- Running with an unclear request ---

--- UNCLEAR REQUEST HANDLER ---
Final Result C: Coordinator could not delegate request: 'I need help.'. Result: Please clarify.


A core component is the `coordinator_router_chain`, which utilizes a `ChatPromptTemplate` to instruct the language model to categorize incoming user requests into one of three categories: 'booker', 'info', or 'unclear'. The output of this router chain is then used by a `RunnableBranch` to delegate the original request to the corresponding handler function. The `RunnableBranch` checks the decision from the language model and directs the request data to either the `booking_handler`, `info_handler`, or `unclear_handler`. The `coordinator_agent` combines these components, first routing the request for a decision and then passing the request to the chosen handler. The final output is extracted from the handler's response.

The code structure mimics a basic multi-agent framework where a central coordinator delegates tasks to specialized agents based on intent.